In [2]:
import ee
import os
import ipywidgets
from ipywidgets import widgets
import ipyleaflet
import geemap
import numpy
import geetools
#import ee.mapclient

ee.Authenticate()

Enter verification code:  4/1AfJohXkXLaKeh79hKaY_drKDgujd3IzF1F1OW1KKtsiU0tlk-YODPlvaBL4



Successfully saved authorization token.


In [3]:
ee.Initialize()

In [4]:
# TASK 1: Load the same pre-downloaded S2 tile on GEE (Hint: Check the name of the downloaded tile)
# and visualize the image using GeeMap

#------------------------------------------------------------------------------------------
# In order to obtain the same mirror Image on the server side from your own downloaded tile
#------------------------------------------------------------------------------------------

#------EXAMPLE----S2B_MSIL2A_20180508T095029_N0207_R079_T34VFJ_20180508T133204.SAFE--------

# First find out the dates; in this case 2018-05-08; so we will find every image inside the GEE image 
# collection that matches the above criterion. 
# In google earth engine, the IDs are stored in this way: the first numeric part represents the sensing date 
# and time, the second numeric part represents the product generation date and time, 
# and the final 6-character string is a unique granule identifier indicating its UTM grid 
# reference (MGRS).

# S2B_MSIL2A_20180508T095029_N0207_R079_T34VFJ_20180508T133204.SAFE
# Date: 2018-05-08 (we will also use filter uptil the next day)
# MGRS_TILE: T34VFJ   (we will skip the 'T')

# Now search usinh the below code:

imageCollection = ee.ImageCollection("COPERNICUS/S2");
image = ee.Image(imageCollection.filterDate('2018-05-08','2018-05-09')\
          .filterMetadata('MGRS_TILE','equals','34VFJ').first());
image

In [16]:
# Visualization parameters 
visParams = {"bands": ['B4', 'B3', 'B2'], "max": 3048, "gamma": 1};
# Map results
import geemap
Map = geemap.Map(center=[40,-100], zoom=4)
Map.centerObject(image,7)
Map.addLayer(image,visParams,'Sentinel-2')
Map

Map(center=[57.22300803612138, 23.565327271913283], controls=(WidgetControl(options=['position', 'transparent_…

In [6]:
# TASK 2: Check Available LUCAS points for the ROI (region of interest)

# Part 2.1: Get the Geometry of the downloaded S2 Image
roi = image.geometry()
roi

ee.Geometry({
  "functionInvocationValue": {
    "functionName": "Image.geometry",
    "arguments": {
      "feature": {
        "functionInvocationValue": {
          "functionName": "Collection.first",
          "arguments": {
            "collection": {
              "functionInvocationValue": {
                "functionName": "Collection.filter",
                "arguments": {
                  "collection": {
                    "functionInvocationValue": {
                      "functionName": "Collection.filter",
                      "arguments": {
                        "collection": {
                          "functionInvocationValue": {
                            "functionName": "ImageCollection.load",
                            "arguments": {
                              "id": {
                                "constantValue": "COPERNICUS/S2"
                              }
                            }
                          }
                        },
                        "filter": {
                          "functionInvocationValue": {
                            "functionName": "Filter.dateRangeContains",
                            "arguments": {
                              "leftValue": {
                                "functionInvocationValue": {
                                  "functionName": "DateRange",
                                  "arguments": {
                                    "end": {
                                      "constantValue": "2018-05-09"
                                    },
                                    "start": {
                                      "constantValue": "2018-05-08"
                                    }
                                  }
                                }
                              },
                              "rightField": {
                                "constantValue": "system:time_start"
                              }
                            }
                          }
                        }
                      }
                    }
                  },
                  "filter": {
                    "functionInvocationValue": {
                      "functionName": "Filter.equals",
                      "arguments": {
                        "leftField": {
                          "constantValue": "MGRS_TILE"
                        },
                        "rightValue": {
                          "constantValue": "34VFJ"
                        }
                      }
                    }
                  }
                }
              }
            }
          }
        }
      }
    }
  }
})

In [8]:
# Part 2.2: Choose the following:
# 1. 'Year' of your downloaded image (HINT: Check Image/Tile ID)
# 2. Correct ROI

point_lucas = ee.FeatureCollection('JRC/LUCAS_HARMO/THLOC/V1')\
               .filterMetadata('year','equals', 2018)\
               .filterBounds(roi).limit(5);

point_lucas.size()

In [9]:
# PART 2.3: Visualize the available LUCAS points
Map.addLayer(point_lucas, {'color':'black'}, 'LUCAS_POINTS')
Map

Map(bottom=10295.0, center=[57.25858752706403, 21.706886316854582], controls=(WidgetControl(options=['position…

In [10]:
# Converting the server side list of points to client side
listOfPoints = point_lucas.toList(point_lucas.size())
n_point = listOfPoints.size()
p=n_point.getInfo()
p

5

In [11]:
# TASK 3: Collect LUCAS street-level images and information for the respective points

# TASK 3.1: Collect and explore LUCAS information/Labels for respective points
Lucas_info=[]
for i in range(p):
    point= ee.Feature(listOfPoints.get(i))
    point_info=point.getInfo()
    Lucas_info.append(point_info)

Lucas_info[0]  # Explore the stored array

# Optional TODO: Store the list in array and explore further

{'type': 'Feature',
 'geometry': {'type': 'Point',
  'coordinates': [24.38430426920031, 57.65970376998848]},
 'id': '000b0000000000002190',
 'properties': {'bio_sample': False,
  'bulk0_10_sample': False,
  'bulk10_20_sample': False,
  'bulk20_30_sample': False,
  'car_ew': 'East',
  'car_latitude': 57.659149169921875,
  'car_longitude': 24.383359909057617,
  'cprn_cando': False,
  'cprn_impervious_perc': 30,
  'cprn_lc': '',
  'cprn_lc1e_brdth': None,
  'cprn_lc1e_next': '',
  'cprn_lc1n': 88,
  'cprn_lc1n_brdth': None,
  'cprn_lc1n_next': '',
  'cprn_lc1s_brdth': None,
  'cprn_lc1s_next': '',
  'cprn_lc1w_brdth': None,
  'cprn_lc1w_next': '',
  'cprn_lc_label': '',
  'cprn_urban': False,
  'cprnc_lc1e': 88,
  'cprnc_lc1s': 88,
  'cprnc_lc1w': 88,
  'crop_residues': False,
  'erosion_cando': False,
  'eunis_complex': '',
  'ex_ante': False,
  'feature_width': '',
  'file_path_gisco_east': 'https://gisco-services.ec.europa.eu/lucas/photos/2018/LV/517/439/51743926E.jpg',
  'file_path_gi

In [12]:
# TASK 3.2 Extract Street-level images for respective points

for k in range(p):
    print(Lucas_info[k]['properties']['file_path_gisco_point'])

https://gisco-services.ec.europa.eu/lucas/photos/2018/LV/517/439/51743926P.jpg
https://gisco-services.ec.europa.eu/lucas/photos/2018/LV/517/839/51783908P.jpg
https://gisco-services.ec.europa.eu/lucas/photos/2018/LV/519/438/51943846P.jpg
https://gisco-services.ec.europa.eu/lucas/photos/2018/LV/519/038/51903838P.jpg
https://gisco-services.ec.europa.eu/lucas/photos/2018/LV/518/239/51823918P.jpg


In [14]:
# TASK 4: Extract 3x3 S2 patches for every LUCAS point with corresponding LUCAS Land cover and Land use label
# Buffer: 20 meters

for i in range(p):
    point= ee.Feature(listOfPoints.get(i))
    point_info=point.getInfo()
    label_lc1=point_info['properties']['lc1']
    label_lu1=point_info['properties']['lu1_label']
    buffer = point.buffer(22)
    patch = image.clip(buffer)
    Map.addLayer(patch, visParams, 'S2 patches'+str(i))
    #file_name = f'/p/project/training2328/sharma6/{i}_{label_lc1}_{label_lu1}.tif'
    #geemap.ee_export_image(patch, filename=file_name, scale=20, region=buffer.geometry(), file_per_band=False)
    print(label_lc1, label_lu1)
Map.centerObject(point_lucas)
Map

# Additional Task: Check the 3x3 patches in QGIS

A11 Residential
A21 Residential
A22 Residential
A22 Road transport
B11 Agriculture (excluding fallow land and kitchen gardens)


Map(bottom=10295.0, center=[57.25858752706403, 21.706886316854582], controls=(WidgetControl(options=['position…